# 00 · Historia, fundamentos y ciclo de vida de Machine Learning

Este laboratorio ya no es solo una introducción rápida: construye la base conceptual que usaremos en toda la ruta.

## Objetivos
- Entender cómo evolucionó ML desde estadística y reconocimiento de patrones hasta ensembles y deep learning.
- Diferenciar supervisado, no supervisado, semi-supervisado, self-supervised, online y reinforcement learning.
- Entender `features`, `target`, inferencia, generalización, overfitting y underfitting.
- Construir un baseline y un experimento reproducible.
- Reconocer data leakage, distribution shift y por qué separar train/validation/test.
- Visualizar el trade-off bias–variance.

## Hitos rápidos
- 1800s–1900s: regresión, probabilidad y estadística.
- 1950s: perceptrón y primeros modelos de aprendizaje.
- 1980s: backpropagation, árboles y sistemas basados en datos.
- 1990s: SVM, bagging y boosting.
- 2000s: Random Forest, datasets masivos y web-scale learning.
- 2010s: deep learning, GPUs y representación aprendida.
- 2020s: foundation models, multimodalidad, AutoML y sistemas híbridos.


## 1. El problema de aprendizaje

En aprendizaje supervisado observamos pares $(x_i,y_i)$ y buscamos una función $f_\theta(x)$ que generalice a datos no vistos. Para regresión minimizamos, por ejemplo, MSE:

$$MSE=\frac{1}{n}\sum_{i=1}^n(y_i-\hat y_i)^2$$

En clasificación usamos pérdidas como log-loss/cross-entropy. La métrica de negocio no siempre coincide con la función de pérdida; un modelo puede optimizar log-loss y evaluarse con recall, F1, AUROC, costo monetario o tiempo ahorrado.

**Ejemplos de uso:** riesgo de deserción estudiantil, priorización de casos, predicción de demanda, fraude, clasificación documental y mantenimiento predictivo.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED=42
X,y=load_breast_cancer(return_X_y=True, as_frame=True)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=SEED,stratify=y)
print(X.shape, X_train.shape, X_test.shape)
X.head()

## 2. Baseline antes de sofisticar

Un baseline responde: **¿mi modelo aporta algo sobre una regla trivial?** En datasets desbalanceados, accuracy puede parecer alta incluso para un modelo inútil. Por eso calcularemos varias métricas.


In [ ]:
rows=[]
for name,model in {
    'majority':DummyClassifier(strategy='most_frequent'),
    'stratified':DummyClassifier(strategy='stratified',random_state=SEED),
    'logistic':LogisticRegression(max_iter=5000)
}.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    proba=model.predict_proba(X_test)[:,1] if hasattr(model,'predict_proba') else pred
    rows.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred,zero_division=0),recall_score(y_test,pred,zero_division=0),f1_score(y_test,pred,zero_division=0),roc_auc_score(y_test,proba)])
pd.DataFrame(rows,columns=['modelo','accuracy','precision','recall','f1','roc_auc']).round(3)

## 3. Train / validation / test

- **Train:** ajusta parámetros.
- **Validation:** selecciona hiperparámetros, features o umbral.
- **Test:** estimación final; no debe participar en decisiones de modelado.

En producción también debemos pensar en un cuarto conjunto conceptual: **datos futuros**, que pueden venir de otra distribución. Para series de tiempo o datos por institución/persona, una partición aleatoria puede ser incorrecta; usaremos `TimeSeriesSplit`, `GroupKFold` y validación temporal en labs posteriores.


## 4. Data leakage: el error que puede hacer parecer excelente a un mal modelo

Leakage ocurre cuando el entrenamiento recibe información que no estaría disponible al momento real de predecir. Ejemplos:
- normalizar usando todo el dataset antes del split;
- usar una variable generada después del evento objetivo;
- tener registros de la misma persona en train y test;
- imputar o seleccionar variables usando también el test;
- construir features temporales mirando hacia el futuro.

**Solución:** pipelines, splits por entidad/tiempo y diseño explícito del instante de inferencia.


In [ ]:
# Demostración de underfitting/overfitting variando complejidad de un árbol
from sklearn.tree import DecisionTreeClassifier
train_scores=[]; test_scores=[]; depths=range(1,16)
for d in depths:
    m=DecisionTreeClassifier(max_depth=d,random_state=SEED).fit(X_train,y_train)
    train_scores.append(m.score(X_train,y_train)); test_scores.append(m.score(X_test,y_test))
plt.plot(depths,train_scores,label='train'); plt.plot(depths,test_scores,label='test')
plt.xlabel('max_depth'); plt.ylabel('accuracy'); plt.legend(); plt.title('Complejidad, bias y variance'); plt.show()

## 5. Bias–variance y generalización

- **Alto bias:** modelo demasiado rígido, falla en train y test.
- **Alta variance:** memoriza train y cae en datos nuevos.
- Regularización, más datos, ensembles, data augmentation y validación ayudan a controlar variance.
- Features más informativas o modelos más expresivos pueden reducir bias.

No existe 'el mejor algoritmo' universal: depende del tamaño, ruido, estructura, restricciones de latencia, interpretabilidad, costo del error y estabilidad de los datos.


## Casos de uso y elección inicial

| Problema | Tipo | Primer baseline razonable | Métricas típicas |
|---|---|---|---|
| Predecir monto | Regresión | Linear/Ridge | MAE, RMSE, R² |
| Riesgo binario | Clasificación | Logistic | Recall, PR-AUC, F1 |
| Segmentación | No supervisado | K-Means | silhouette + validación de negocio |
| Fraude raro | Clasificación desbalanceada | Logistic + class weight | PR-AUC, recall@precision |
| Demanda futura | Time series | naive/seasonal naive | MAE, MAPE, RMSE |
| Anomalías | Unsupervised/semi-supervised | z-score/IsolationForest | precision@k + revisión humana |

## Ejercicios
1. Cambia la proporción de test y compara incertidumbre de métricas.
2. Sustituye el árbol por KNN y observa cómo cambia el overfitting.
3. Diseña un ejemplo de leakage en un sistema de riesgo estudiantil o gubernamental.
4. Explica por qué un accuracy de 99% puede ser pésimo en fraude.
5. Repite el experimento con 10 semillas y grafica distribución del score.

## Qué sigue
En los próximos labs veremos preprocessing, regresión/clasificación, árboles, SVM/KNN/NB, clustering, redes neuronales, boosting, imbalance, explicabilidad, causalidad, forecasting, Spark y MLOps.
